In [ ]:
import pandas as pd
import time
import joblib
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier, plot_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay


In [ ]:

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_parquet("/ec/vol/centaur/pajm/training_data2.parquet")

print("Loaded data shape:", df.shape)
print("Columns:", list(df.columns))


In [ ]:

# -----------------------------
# DEFINE FEATURES & TARGET
# -----------------------------
target_col = "AF"
feature_cols = [c for c in df.columns if c not in ["AF"]]

X = df[feature_cols]
y = df[target_col]


In [ ]:

# -----------------------------
# TRAIN / TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:

# -----------------------------
# TRAIN CLASSIFIER
# -----------------------------
print("Training XGBoost Binary Classifier...")
start_time = time.time()

model = XGBClassifier(
    objective="binary:logistic",
    tree_method="hist",
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
#    subsample=0.8,
#    colsample_bytree=0.8,
    eval_metric="logloss",
#    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

end_time = time.time()
print(f"✅ Training completed in {end_time - start_time:.2f} seconds")


In [ ]:

# -----------------------------
# EVALUATE MODEL
# -----------------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))

roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.3f}")

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:

# -----------------------------
# PLOT ROC CURVE & FEATURE IMPORTANCE
# -----------------------------
plt.figure(figsize=(6, 5))
RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.title("ROC Curve")
plt.savefig("/home/pajm/python/POFINABOX/POF_ROC.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
plot_importance(model, max_num_features=20, importance_type="gain")
plt.title("Feature Importance (Gain)")
plt.tight_layout()
plt.savefig("/home/pajm/python/POFINABOX/POF_importance.png", dpi=300)
plt.close()


In [ ]:

# -----------------------------
# SAVE MODEL
# -----------------------------
out_model = "/home/pajm/python/POFINABOX/POF_model.joblib"
joblib.dump(model, out_model, compress=3)
print(f"Model saved → {out_model}")